In [ ]:
# función de limpieza que se usarát ambién con el test.csv
# ============================
# 1. LIMPIEZA DE RAM
# ============================
def clean_ram(df):
    df["Ram"] = df["Ram"].str.replace("GB", "").astype(int)
    return df

# ============================
# 2. LIMPIEZA DE PESO
# ============================
def clean_weight(df):
    df["Weight"] = (
        df["Weight"]
        .astype(str)              # Convertir todo a string
        .str.replace("kg", "")    # Quitar "kg" si existe
        .str.strip()              # Quitar espacios
    )
    
    # Convertir a float de forma segura
    df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
    
    return df


# ============================
# 3. PARSEAR MEMORY
# ============================
def parse_memory(df):
    # Normalizar texto
    mem = df["Memory"].str.replace("GB", "").str.replace("TB", "000")

    # Separar si hay dos unidades (ej: "256 SSD + 1TB HDD")
    parts = mem.str.split("+", expand=True)

    # Función auxiliar
    def extract_capacity(text):
        if text is None:
            return 0
        text = text.strip()
        if "SSD" in text:
            return int(re.findall(r"\d+", text)[0])
        if "HDD" in text:
            return int(re.findall(r"\d+", text)[0])
        if "Flash" in text:
            return int(re.findall(r"\d+", text)[0])
        if "Hybrid" in text:
            return int(re.findall(r"\d+", text)[0])
        return 0

    df["SSD"] = parts[0].apply(lambda x: extract_capacity(x) if "SSD" in str(x) else 0)
    df["HDD"] = parts[0].apply(lambda x: extract_capacity(x) if "HDD" in str(x) else 0)
    df["Flash"] = parts[0].apply(lambda x: extract_capacity(x) if "Flash" in str(x) else 0)
    df["Hybrid"] = parts[0].apply(lambda x: extract_capacity(x) if "Hybrid" in str(x) else 0)

    # Si hay segunda parte
    if parts.shape[1] > 1:
        df["SSD"] += parts[1].apply(lambda x: extract_capacity(x) if "SSD" in str(x) else 0)
        df["HDD"] += parts[1].apply(lambda x: extract_capacity(x) if "HDD" in str(x) else 0)
        df["Flash"] += parts[1].apply(lambda x: extract_capacity(x) if "Flash" in str(x) else 0)
        df["Hybrid"] += parts[1].apply(lambda x: extract_capacity(x) if "Hybrid" in str(x) else 0)

    return df

# ============================
# 4. PARSEAR CPU
# ============================
def parse_cpu(df):
    df["Cpu_brand"] = df["Cpu"].str.split().str[0]
    df["Cpu_model"] = df["Cpu"].str.extract(r"(i3|i5|i7|i9|Ryzen\s?\d+)")
    return df

# ============================
# 5. PARSEAR GPU
# ============================
def parse_gpu(df):
    df["Gpu_brand"] = df["Gpu"].str.split().str[0]
    return df
# ============================
# 6. PARSEAR SCREEN RESOLUTION
# ============================
def parse_screen_resolution(df):
    df["Touchscreen"] = df["ScreenResolution"].str.contains("Touchscreen").astype(int)

    # Extraer resolución
    df["Resolution_X"] = df["ScreenResolution"].str.extract(r"(\d+)x").astype(int)
    df["Resolution_Y"] = df["ScreenResolution"].str.extract(r"x(\d+)").astype(int)

    return df

# ============================
# 7. FUNCIÓN PRINCIPAL
# ============================
def clean_dataset(df):
    df = df.copy()

    df = clean_ram(df)
    df = clean_weight(df)
    df = parse_memory(df)
    df = parse_cpu(df)
    df = parse_gpu(df)
    df = parse_screen_resolution(df)

    # Eliminar columnas originales que ya no sirven
    df = df.drop(columns=["Memory", "Cpu", "Gpu", "ScreenResolution"])

    return df